In [ ]:
# Nitsa Saint Fort
# 11/6/25
# CAP 4770
# Assignment 4
import os
import numpy as np
import pandas as pd
import matplotlib.image as mpimg
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from google.colab import files
import time

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
data_path = "UTKFace"

In [ ]:
uploaded = files.upload()
zip_file = list(uploaded.keys())[0]
import zipfile
with zipfile.ZipFile(zip_file, 'r') as z:
  z.extractall()
  data_path = "UTKFace"
  IMG_Size = 32
  batch_size = 4
  images = []
Ages = []
Genders = []
Ethnicities = []

print("Loading images...")
for filename in os.listdir(data_path):
    parts = filename.split('_')
    if len(parts) != 4:
        continue

    age, gender, ethnicity, _ = parts

    try:
        age = int(age)
        gender = int(gender)
        ethnicity = int(ethnicity)
    except ValueError:
        continue

    img_path = os.path.join(data_path, filename)

    # Read image using matplotlib
    try:
        image = mpimg.imread(img_path)
    except Exception as e:
        print(f"Skipping {filename}: {e}")
        continue

    # Resize all images to (40, 40)
    image = cv2.resize(image, (40, 40))

    # Ensure image is RGB and normalized
    if len(image.shape) == 2:  # Grayscale
        image = np.stack([image]*3, axis=-1)

    Ages.append(age)
    Genders.append(gender)
    Ethnicities.append(ethnicity)
    images.append(image)

print(f"Loaded {len(images)} images")

if len(images) == 0:
    print("ERROR: No images loaded! Check your data path.")
    exit()

In [ ]:
print(os.listdir(data_path))

FileNotFoundError: [Errno 2] No such file or directory: 'UTKFace'

In [ ]:
# Convert to numpy array and prepare for PyTorch
images_np = np.stack(images, axis=0)  # shape: (N, H, W, 3)
# Normalize to [0, 1] - handle both uint8 and float formats
if images_np.max() > 1.0:
    images_np = images_np / 255.0

# Convert from (N, H, W, 3) → (N, 3, H, W) for PyTorch
images_np = np.transpose(images_np, (0, 3, 1, 2))

# Convert to torch tensors
images_tensor = torch.tensor(images_np, dtype=torch.float32)
genders_tensor = torch.tensor(Genders, dtype=torch.long)

print(f"Images shape: {images_tensor.shape}")
print(f"Genders shape: {genders_tensor.shape}")

In [ ]:
# Split into train and test sets
x_train, x_test, y_train, y_test = train_test_split(
    images_tensor, genders_tensor, test_size=0.15, random_state=42
)

print(f"Training set size: {x_train.shape[0]}")
print(f"Test set size: {x_test.shape[0]}")

In [ ]:
train_data = TensorDataset(x_train, y_train)
test_data = TensorDataset(x_test, y_test)
train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = DataLoader(test_data, batch_size=8, shuffle=True)

In [ ]:
# Build model
model = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1),      # 40x40x3 → 40x40x8
    nn.ReLU(),
    nn.MaxPool2d(2),                     # 40x40x8 → 25x25x8

    nn.Conv2d(8, 16, 3, padding=1),     # 25x25x8 → 25x25x16
    nn.ReLU(),
    nn.MaxPool2d(2),                     # 25x25x16 → 12x12x16

    nn.Conv2d(16, 64, 3, padding=1),    # 12x12x16 → 12x12x64
    nn.ReLU(),
    nn.MaxPool2d(2),                     # 12x12x64 → 6x6x64

    nn.Conv2d(64, 128, 3, padding=1),   # 6x6x64 → 6x6x128
    nn.ReLU(),
    nn.MaxPool2d(2),                     # 6x6x128 → 3x3x128

    nn.Flatten(),
    nn.Linear(3*3*128, 40),
    nn.ReLU(),
    nn.Linear(40, 2),  # 2 classes for binary gender classification
    nn.Softmax(dim=1)
)


In [ ]:
model = model.to(device)
print(f"Model moved to: {device}")

Model moved to: cpu


In [ ]:
# Define loss function and optimizer
loss_fn = nn.CrossEntropyLoss()  # Better for classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Training loop
num_epochs = 40

print("\nStarting training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")


Starting training...


NameError: name 'train_loader' is not defined

In [ ]:
# Testing phase
print("\nEvaluating on test set...")
model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        test_loss += loss.item()

        # Get predictions
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
print(f"Test Loss: {test_loss/len(test_loader):.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
images_numpy = np.stack(images,axis = 0) # (23,708, 40, 40, 3), (N,H,W,C)
images_numpy = np.transpose(images_numpy,(0,3,1,2))
images_tensor = torch.tensor(images_numpy, dtype=torch.float32) # convert to tensor
Ages_tensor = torch.tensor(images_numpy, dtype=torch.float32)
Genders_tensor = torch.tensor(images_numpy, dtype=torch.float32)
#for Tensor ---> (N,C,H,W)

In [ ]:
# Build the model
# 40 x 40 x 3
My_model = nn.Sequential(nn.Conv2d(3,8,3), #48x48x3
                       nn.ReLU(),
                       nn.MaxPool(2),#24x24x8
                       nn.Conv2d(), #22x22x32
                       nn.ReLU(),
                       nn.MaxPool(2), # 11x11x32
                       nn.Conv2d(32,64,3)), # 9x9x64
                       nn.ReLU(),
                       nn.MaxPool(2), # 4x4x64
                       nn.Flatten(), #1024
                       nn.Linear(1024,1),
                       nn.Sigmoid()


In [ ]:
# Training
My_model.fit(train_data)

In [ ]:
My_model.fit(x_train, y_train, epochs=40)

In [ ]:
My_model.evaluate(x_test,y_test)

In [ ]:
y_pred = My_model.predict(x_test)
y_pred[:5]

In [ ]:
# DNN model
# 30 x 30 x 3
My_model2 = nn.Sequential(nn.Conv2d(3,8,3), #36x36x3
 nn.ReLU(),
 nn.MaxPool(2),#24x24x8
 nn.Conv2d(), #22x22x32
 nn.ReLU(),
 nn.MaxPool(2), # 11x11x32
 nn.Conv2d(32,64,3)), # 9x9x64
nn.ReLU(),
nn.MaxPool(2), # 4x4x64
nn.Flatten(), #1024
nn.Linear(1024,1),
nn.Sigmoid()

In [ ]:
My_model2.fit(train_data)

In [ ]:
My_model2.fit(x_train, y_train, epochs=30)

In [ ]:
My_model2.evaluate(x_test,y_test)

In [ ]:
y_pred = My_model2.predict(x_test)
y_pred[:5]

4.

Using the CNN can improve accuracy but have a slower runtime. Using the DNN can have a decreased accuracy but a faster runtime.

When I try to run the program, it takes a long for it to compile. I switched to colab pro with my personal account and only some of the images are upload and the program keeps getting stuck on the 6_1_2_20161219190053203.jpg.chip.jpg.